# ANOVA Analysis of *Spondias mombin* Antibacterial Zone of Inhibition Data

This notebook performs one-way ANOVA, Tukey HSD post-hoc comparisons, two-way ANOVA and
organism-wise ANOVA on *S. mombin* zone of inhibition (IZD) measurements.

## Dataset
- `Smombin_zones_anova_file.csv`
- 20 bacterial isolates
- 3 extract concentrations (400, 200 and 100 mg/mL)
- 3 zone-of-inhibition replicates per concentration
- The solvent-system control is excluded because all its zones are 0 mm (no variance)

## Analysis
- One-way ANOVA: effect of concentration on mean zone of inhibition
- Tukey HSD: post-hoc pairwise comparisons
- Assumption checks: normality (Shapiro-Wilk on residuals) and homogeneity of variance (Levene)
- Two-way ANOVA: concentration x Gram status
- Organism-wise one-way ANOVA (isolates with no inhibition or zero within-group variance are reported as n/a)
- Blocked model (concentration + organism), because the readings are nested within isolates

**Significance convention:** `*` = p < 0.05 (single level; no ** or ***).

## How to run
```bash
pip install -r requirements.txt
jupyter notebook
```
Input: `data/processed/Smombin_zones_anova_file.csv`. Outputs are written to `results/`.

Corresponding author: Abdullahi Abubakar Sadiq (abubakarabdulsadiq@gmail.com)

In [ ]:
# =========================================================
# S. MOMBIN IZD: STATISTICAL ANALYSIS OF ANTIMICROBIAL ZONE DIAMETERS
# =========================================================
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import shapiro, levene

from pathlib import Path
# Repository paths (works when run from the repository root or from notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

ALPHA = 0.05   # * = p < 0.05

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
df = pd.read_csv(DATA / "processed" / "Smombin_zones_anova_file.csv")

# Data types
df["concentration_mg_ml"] = df["concentration_mg_ml"].astype(str)
df["gram_status"] = df["gram_status"].astype("category")
df["organism"] = df["organism"].astype("category")

# ---------------------------------------------------------
# Utility functions
# ---------------------------------------------------------
def add_eta_squared(anova_table):
    a = anova_table.copy()
    a["eta_sq"] = a["sum_sq"] / a["sum_sq"].sum()
    return a

def format_p(p):
    if pd.isna(p):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def star(p):
    return "*" if (not pd.isna(p) and p < ALPHA) else ""

def add_sig_columns(anova_table):
    a = anova_table.copy()
    a["p_formatted"] = a["PR(>F)"].apply(format_p)
    a["Sig"] = a["PR(>F)"].apply(star)
    return a

print(df.shape, df["organism"].nunique(), "isolates;",
      sorted(df["concentration_mg_ml"].unique()), "mg/mL")

In [ ]:
# ---------------------------------------------------------
# 1. ONE-WAY ANOVA (Concentration), pooled across isolates
# ---------------------------------------------------------
model_oneway = ols("zone_mm ~ C(concentration_mg_ml)", data=df).fit()
anova_oneway = add_sig_columns(add_eta_squared(sm.stats.anova_lm(model_oneway, typ=2)))

# Assumption checks
shapiro_p = shapiro(model_oneway.resid).pvalue
groups = [df.loc[df["concentration_mg_ml"] == c, "zone_mm"]
          for c in df["concentration_mg_ml"].unique()]
levene_p = levene(*groups).pvalue

# Post-hoc: Tukey HSD
tukey_oneway = pairwise_tukeyhsd(
    endog=df["zone_mm"],
    groups=df["concentration_mg_ml"],
    alpha=ALPHA
)
tukey_df = pd.DataFrame(tukey_oneway._results_table.data[1:],
                        columns=tukey_oneway._results_table.data[0])
tukey_df["p_formatted"] = tukey_df["p-adj"].apply(format_p)
tukey_df["Sig"] = tukey_df["p-adj"].apply(star)

assumptions_df = pd.DataFrame({
    "Test": ["Shapiro-Wilk (residuals)", "Levene (equal variances)"],
    "p_value": [shapiro_p, levene_p],
    "p_formatted": [format_p(shapiro_p), format_p(levene_p)],
})

anova_oneway.to_csv(RESULTS / "S_mombin_IZD_anova_oneway_full.csv")
tukey_df.to_csv(RESULTS / "S_mombin_IZD_tukey_oneway.csv", index=False)
assumptions_df.to_csv(RESULTS / "S_mombin_IZD_assumptions.csv", index=False)

print("One-way ANOVA (concentration)")
print(anova_oneway.round(4).to_string())
print("\nAssumption checks")
print(assumptions_df.to_string(index=False))
print("\nTukey HSD")
print(tukey_oneway.summary())

In [ ]:
# ---------------------------------------------------------
# 2. TWO-WAY ANOVA (Concentration x Gram status)
# ---------------------------------------------------------
model_twoway = ols("zone_mm ~ C(concentration_mg_ml) * C(gram_status)", data=df).fit()
anova_twoway = add_sig_columns(add_eta_squared(sm.stats.anova_lm(model_twoway, typ=2)))
anova_twoway.to_csv(RESULTS / "S_mombin_IZD_anova_twoway_full.csv")
print(anova_twoway.round(4).to_string())

In [ ]:
# ---------------------------------------------------------
# 3. BLOCKED MODEL: concentration + organism
#    (readings are nested within isolates, so isolate is a blocking factor)
# ---------------------------------------------------------
model_block = ols("zone_mm ~ C(concentration_mg_ml) + C(organism)", data=df).fit()
anova_block = add_sig_columns(add_eta_squared(sm.stats.anova_lm(model_block, typ=2)))
anova_block.to_csv(RESULTS / "S_mombin_IZD_anova_blocked.csv")
print(anova_block.round(4).to_string())

In [ ]:
# ---------------------------------------------------------
# 4. ORGANISM-WISE ONE-WAY ANOVA (400, 200, 100 mg/mL)
#    * = p < 0.05
# ---------------------------------------------------------
rows = []
for org in df["organism"].cat.categories:
    sub = df[df["organism"] == org]

    # No variation at all (e.g. every zone = 0): ANOVA is undefined
    if sub["zone_mm"].nunique() == 1:
        rows.append({"Organism": org, "F": np.nan, "p_value": np.nan,
                     "p_formatted": "n/a", "Sig": "",
                     "Note": "No inhibition at any concentration (all zones = 0)"})
        continue

    model_org = ols("zone_mm ~ C(concentration_mg_ml)", data=sub).fit()
    aov = sm.stats.anova_lm(model_org, typ=2)

    # Zero within-group variance: F and p are not interpretable
    if aov.loc["Residual", "sum_sq"] < 1e-9:
        rows.append({"Organism": org, "F": np.nan, "p_value": np.nan,
                     "p_formatted": "n/a", "Sig": "",
                     "Note": "Zero within-group variance; report descriptively"})
        continue

    p = aov.loc["C(concentration_mg_ml)", "PR(>F)"]
    rows.append({"Organism": org,
                 "F": aov.loc["C(concentration_mg_ml)", "F"],
                 "p_value": p, "p_formatted": format_p(p),
                 "Sig": star(p), "Note": ""})

organism_anova_df = pd.DataFrame(rows)
organism_anova_df.to_csv(RESULTS / "S_mombin_IZD_organism_wise_anova.csv", index=False)
print(organism_anova_df.round(4).to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# 5. FIGURES
# ---------------------------------------------------------
df_plot = df.copy()
df_plot["concentration_mg_ml"] = df_plot["concentration_mg_ml"].astype(float)

# 5a. Dose-response (all isolates pooled)
summary_plot = (df_plot.groupby("concentration_mg_ml")["zone_mm"]
                .agg(["mean", "sem"]).reset_index())

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(summary_plot["concentration_mg_ml"], summary_plot["mean"],
            yerr=summary_plot["sem"], marker="o", linestyle="-", capsize=4)
ax.set_xlabel("Concentration (mg/mL)")
ax.set_ylabel("Zone of inhibition (mm)")
ax.set_title("S. mombin IZD dose-response relationship")
fig.tight_layout()
fig.savefig(RESULTS / "S_mombin_IZD_dose_response.png", dpi=300)
plt.show()

# 5b. Dose-response by Gram status
fig, ax = plt.subplots(figsize=(6, 4))
for g in df_plot["gram_status"].cat.categories:
    s = (df_plot[df_plot["gram_status"] == g]
         .groupby("concentration_mg_ml")["zone_mm"]
         .agg(["mean", "sem"]).reset_index())
    ax.errorbar(s["concentration_mg_ml"], s["mean"], yerr=s["sem"],
                marker="o", linestyle="-", capsize=4, label=g)
ax.set_xlabel("Concentration (mg/mL)")
ax.set_ylabel("Zone of inhibition (mm)")
ax.set_title("S. mombin IZD dose-response by Gram status")
ax.legend(title="Gram status")
fig.tight_layout()
fig.savefig(RESULTS / "S_mombin_IZD_dose_response_by_gram.png", dpi=300)
plt.show()

print("Analysis complete. Tables (.csv) and figures (.png) saved to the working folder.")